In [ ]:

# ==================================================
# 侍テラコヤ 自動ログイン・プロフィール更新
# 実行環境：Google Colab
# 使用ライブラリ：Selenium
# ==================================================

# --------------------------------------------------
# 1. ライブラリのインストール
# --------------------------------------------------

%pip -q install selenium

# --------------------------------------------------
# 2. ライブラリの読み込み
# --------------------------------------------------

import os
import getpass

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

# --------------------------------------------------
# 3. 設定
# --------------------------------------------------

LOGIN_URL = "https://terakoya.sejuku.net/register"

PROFILE_URL = "https://terakoya.sejuku.net/account/profile"

NEW_INTRODUCTION = (
    "プログラミング学習中です！"
    "今はスクレイピングに挑戦しています！"
)

# --------------------------------------------------
# 4. ログイン情報の入力
# --------------------------------------------------

email_address = input(
    "メールアドレスを入力してください："
).strip()

password = getpass.getpass(
    "パスワードを入力してください："
)

# --------------------------------------------------
# 5. Chromeの設定
# --------------------------------------------------

# Snap版Chromium用のChromeDriverを回避
os.environ["SE_SKIP_DRIVER_IN_PATH"] = "true"

chrome_options = Options()

chrome_options.binary_location = "/usr/bin/google-chrome"

chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--window-size=1920,1080")

# Chromeを起動
chrome_driver = webdriver.Chrome(
    options=chrome_options
)

wait = WebDriverWait(chrome_driver, 20)

print("\nChromeの起動に成功しました")

# --------------------------------------------------
# 6. ログインフォームを取得する関数
# --------------------------------------------------

def get_login_form(driver):

    login_button = driver.find_element(
        By.XPATH,
        "//button[normalize-space()='ログイン']"
    )

    form_data = driver.execute_script(
        """
        const button = arguments[0];

        let parent = button.parentElement;

        while (parent) {

            const email = parent.querySelector(
                'input[name="email"]'
            );

            const password = parent.querySelector(
                'input[name="password"]'
            );

            if (email && password) {
                return [email, password];
            }

            parent = parent.parentElement;
        }

        return null;
        """,
        login_button
    )

    if not form_data:
        raise RuntimeError(
            "ログインフォームが見つかりません"
        )

    return (
        form_data[0],
        form_data[1],
        login_button
    )

# --------------------------------------------------
# 7. 侍テラコヤへアクセス
# --------------------------------------------------

chrome_driver.get(LOGIN_URL)

print("侍テラコヤにアクセスしました")

# --------------------------------------------------
# 8. ログイン画面を開く
# --------------------------------------------------

login_link = wait.until(
    EC.element_to_be_clickable(
        (
            By.XPATH,
            "//header//div[normalize-space()='ログイン']"
        )
    )
)

login_link.click()

print("ログイン画面を開きました")

# --------------------------------------------------
# 9. ログインフォームを取得
# --------------------------------------------------

wait.until(
    EC.presence_of_element_located(
        (
            By.XPATH,
            "//button[normalize-space()='ログイン']"
        )
    )
)

email_input, password_input, login_button = (
    get_login_form(chrome_driver)
)

print("ログインフォームを取得しました")

# --------------------------------------------------
# 10. ログイン情報を入力
# --------------------------------------------------

email_input.clear()
email_input.send_keys(email_address)

password_input.clear()
password_input.send_keys(password)

print("ログイン情報を入力しました")

# --------------------------------------------------
# 11. 入力内容を確認
# --------------------------------------------------

email_check = (
    email_input.get_attribute("value")
    == email_address
)

password_check = (
    password_input.get_attribute("value")
    == password
)

print("メールアドレス入力確認：", email_check)
print("パスワード入力確認：", password_check)

if not email_check or not password_check:
    raise RuntimeError(
        "ログイン情報の入力に失敗しました"
    )

# --------------------------------------------------
# 12. ログインボタンをクリック
# --------------------------------------------------

wait.until(
    lambda driver: login_button.is_enabled()
)

login_button.click()

print("ログインボタンをクリックしました")

# --------------------------------------------------
# 13. ログイン成功を確認
# --------------------------------------------------

try:

    wait.until(
        EC.url_contains("/home")
    )

    print("ログインに成功しました")

except TimeoutException:

    chrome_driver.save_screenshot(
        "login_error.png"
    )

    raise RuntimeError(
        "ログイン完了を確認できませんでした"
    )

print("現在のURL：", chrome_driver.current_url)

# --------------------------------------------------
# 14. プロフィールページへ移動
# --------------------------------------------------

chrome_driver.get(PROFILE_URL)

wait.until(
    EC.element_to_be_clickable(
        (
            By.XPATH,
            "//button[normalize-space()='編集']"
        )
    )
)

print("\nプロフィールページに移動しました")

# --------------------------------------------------
# 15. 編集ボタンをクリック
# --------------------------------------------------

edit_button = chrome_driver.find_element(
    By.XPATH,
    "//button[normalize-space()='編集']"
)

edit_button.click()

print("編集ボタンをクリックしました")

# --------------------------------------------------
# 16. 自己紹介欄を取得
# --------------------------------------------------

wait.until(
    lambda driver: len(
        driver.find_elements(
            By.TAG_NAME,
            "textarea"
        )
    ) == 2
)

textareas = chrome_driver.find_elements(
    By.TAG_NAME,
    "textarea"
)

# 動作確認済みの1番目のtextarea
introduction_input = textareas[0]

# 自己紹介の項目名を確認
label_found = chrome_driver.execute_script(
    """
    let element = arguments[0];

    for (let i = 0; i < 6; i++) {

        if (!element) break;

        const text = element.innerText || "";

        if (text.includes("自己紹介")) {
            return true;
        }

        element = element.parentElement;
    }

    return false;
    """,
    introduction_input
)

if not label_found:
    raise RuntimeError(
        "自己紹介欄を特定できませんでした"
    )

print("自己紹介欄を取得しました")

# --------------------------------------------------
# 17. 自己紹介文を入力
# --------------------------------------------------

introduction_input.clear()

introduction_input.send_keys(
    NEW_INTRODUCTION
)

# 入力内容を確認
input_check = (
    introduction_input.get_attribute("value")
    == NEW_INTRODUCTION
)

print("自己紹介文の入力確認：", input_check)

if not input_check:
    raise RuntimeError(
        "自己紹介文の入力に失敗しました"
    )

print("自己紹介文を入力しました")

# --------------------------------------------------
# 18. 更新ボタンをクリック
# --------------------------------------------------

update_button = wait.until(
    EC.element_to_be_clickable(
        (
            By.XPATH,
            "//button[normalize-space()='更新する']"
        )
    )
)

update_button.click()

print("更新ボタンをクリックしました")

# 編集画面が終了するまで待機
wait.until(
    EC.element_to_be_clickable(
        (
            By.XPATH,
            "//button[normalize-space()='編集']"
        )
    )
)

print("編集画面が終了しました")

# --------------------------------------------------
# 19. プロフィールページを再読み込み
# --------------------------------------------------

chrome_driver.get(PROFILE_URL)

print("プロフィールページを再読み込みしました")

# --------------------------------------------------
# 20. 再度編集画面を開く
# --------------------------------------------------

edit_button = wait.until(
    EC.element_to_be_clickable(
        (
            By.XPATH,
            "//button[normalize-space()='編集']"
        )
    )
)

edit_button.click()

print("保存確認のため編集画面を開きました")

# --------------------------------------------------
# 21. 保存された自己紹介文を確認
# --------------------------------------------------

wait.until(
    lambda driver: len(
        driver.find_elements(
            By.TAG_NAME,
            "textarea"
        )
    ) == 2
)

textareas = chrome_driver.find_elements(
    By.TAG_NAME,
    "textarea"
)

saved_text = textareas[0].get_attribute("value")

saved_check = (
    saved_text == NEW_INTRODUCTION
)

print("\n===== 保存内容の確認 =====")

print("保存された自己紹介文：", saved_text)

print("指定した文章との一致：", saved_check)

# --------------------------------------------------
# 22. 最終結果
# --------------------------------------------------

print("\n================================")

if saved_check:

    print("侍テラコヤ 自動更新成功")

    print("自己紹介文の保存を確認しました")

else:

    print("自己紹介文の保存を確認できませんでした")

print("================================")

# --------------------------------------------------
# 23. ブラウザの終了
# --------------------------------------------------

# 動作確認後に終了する場合は、
# 以下のコメントを外してください。

# chrome_driver.quit()
